# LangGraph basics with toy graphs

Five tiny graphs, one concept each, using the same LangGraph features the incident-triage project uses. No Azure, no LLM, nothing from the project: just the ideas.

| # | Concept | Office picture | Used in the project for |
|---|---|---|---|
| 1 | State, nodes, edges | Whiteboard, workers, arrows | The whole agent flow |
| 2 | Parallel and reducers | Three people work at once, notes are merged | Logs, Metrics, Change agents |
| 3 | Conditional edges | A decision diamond | The Diagnosis loop |
| 4 | Interrupt and resume | "Stop until the manager signs" | The approval gate |
| 5 | Checkpointer and thread id | A save game | Pausing and resuming a run |

**To run it:** Python 3.12 and `pip install langgraph`. Or from the repo root: `pip install -e "backend[dev]"`.

## 1. State, nodes and edges

- **State** is a dict every node can read. Here it has two keys.
- A **node** is a plain function. It receives the state and returns *only the keys it wants to change*.
- An **edge** says which node runs next. `START` and `END` mark the ends.

In [1]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    name: str
    greeting: str


def pick_greeting(state: State) -> dict:
    return {"greeting": f"hello {state['name']}"}      # returns only what it changes


def shout(state: State) -> dict:
    return {"greeting": state["greeting"].upper() + "!"}


g = StateGraph(State)
g.add_node("pick_greeting", pick_greeting)
g.add_node("shout", shout)
g.add_edge(START, "pick_greeting")
g.add_edge("pick_greeting", "shout")
g.add_edge("shout", END)
app = g.compile()

print(app.invoke({"name": "sam"}))      # the final state: input keys plus whatever the nodes added

{'name': 'sam', 'greeting': 'HELLO SAM!'}


## 2. Parallel nodes and reducers

Give one node several outgoing edges and they all run **at the same time**. A node with several incoming edges waits for **all** of them.

There is a catch: if three nodes write the *same key* at once, LangGraph needs to know how to combine them. That rule is a **reducer**. Without one, it refuses. This is why the project's `findings` key has a merge rule.

In [2]:
from langgraph.errors import InvalidUpdateError


def merge(a: dict, b: dict) -> dict:           # the reducer: combine two dicts
    return {**a, **b}


class Plain(TypedDict, total=False):
    n: int
    results: dict                              # no reducer
    total: int


class Merged(TypedDict, total=False):
    n: int
    results: Annotated[dict, merge]            # with a reducer
    total: int


def build(state_type):
    def double(s): return {"results": {"double": s["n"] * 2}}
    def square(s): return {"results": {"square": s["n"] ** 2}}
    def negate(s): return {"results": {"negate": -s["n"]}}
    def total(s):  return {"total": sum(s["results"].values())}

    g = StateGraph(state_type)
    for fn in (double, square, negate, total):
        g.add_node(fn.__name__, fn)
    for worker in ("double", "square", "negate"):
        g.add_edge(START, worker)              # fan out: all three start together
    g.add_edge(["double", "square", "negate"], "total")   # fan in: total waits for all three
    g.add_edge("total", END)
    return g.compile()


try:
    build(Plain).invoke({"n": 3})
except InvalidUpdateError as e:
    print("without a reducer:", type(e).__name__)

print("with a reducer:   ", build(Merged).invoke({"n": 3}))

without a reducer: InvalidUpdateError
with a reducer:    {'n': 3, 'results': {'double': 6, 'negate': -3, 'square': 9}, 'total': 12}


## 3. Conditional edges (a loop with a cap)

A **conditional edge** runs a small function after a node and uses its answer to pick the next node. Pointing the answer back at an earlier node makes a loop. The cap (`attempts >= 3`) is what stops it looping forever, like `max_rounds` in the project.

In [3]:
class Draft(TypedDict, total=False):
    text: str
    attempts: int


def improve(state: Draft) -> dict:
    return {"text": state["text"] + "!", "attempts": state.get("attempts", 0) + 1}


def good_enough(state: Draft) -> str:
    done = len(state["text"]) >= 6 or state["attempts"] >= 3
    return "stop" if done else "again"


g = StateGraph(Draft)
g.add_node("improve", improve)
g.add_edge(START, "improve")
g.add_conditional_edges("improve", good_enough, {"again": "improve", "stop": END})
loop = g.compile()

print(loop.invoke({"text": "hello"}))     # 6 characters after one pass: stops because it is long enough
print(loop.invoke({"text": ""}))          # never gets long enough: stops because of the cap (3 attempts)

{'text': 'hello!', 'attempts': 1}
{'text': '!!!', 'attempts': 3}


## 4. Interrupt and resume (a human in the loop)

`interrupt(...)` pauses the graph and hands a message to the caller. The graph stops and its state is saved. Later the caller resumes it with `Command(resume=...)` and **that value becomes the return value of `interrupt`**.

Two rules worth remembering, both from the project:
1. It needs a **checkpointer**, which is where the paused state is saved.
2. When it resumes, the node starts again **from the top**. So put `interrupt` first in the node, before anything that must not run twice.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt


class Order(TypedDict, total=False):
    proposal: str
    decision: str
    result: str


def propose(state: Order) -> dict:
    return {"proposal": "refund 20 dollars"}


def gate(state: Order) -> dict:
    decision = interrupt({"please review": state["proposal"]})   # first statement: pauses here
    return {"decision": decision}


def apply(state: Order) -> dict:
    return {"result": f"applied: {state['proposal']}"}


def skip(state: Order) -> dict:
    return {"result": "nothing was done"}


g = StateGraph(Order)
for fn in (propose, gate, apply, skip):
    g.add_node(fn.__name__, fn)
g.add_edge(START, "propose")
g.add_edge("propose", "gate")
g.add_conditional_edges("gate", lambda s: "yes" if s["decision"] == "approve" else "no", {"yes": "apply", "no": "skip"})
g.add_edge("apply", END)
g.add_edge("skip", END)
review = g.compile(checkpointer=InMemorySaver())      # the checkpointer is required for interrupt

config = {"configurable": {"thread_id": "order-1"}}
first = review.invoke({}, config)
print("paused. message for the human:", first["__interrupt__"][0].value)
print("next node waiting:", review.get_state(config).next)

final = review.invoke(Command(resume="approve"), config)   # the human answers
print("after approve:", final["result"])

paused. message for the human: {'please review': 'refund 20 dollars'}
next node waiting: ('gate',)
after approve: applied: refund 20 dollars


## 5. Checkpointer and thread id

The checkpointer saves the state under a **thread id**. Same id means the same run. A different id is a separate run with its own saved state. In the project, each incident run uses its own thread id, so many runs can wait for approval at once, and rejecting one does not affect another.

In [5]:
a = {"configurable": {"thread_id": "order-A"}}
b = {"configurable": {"thread_id": "order-B"}}

review.invoke({}, a)          # both runs pause at the gate
review.invoke({}, b)
print("A is waiting at:", review.get_state(a).next)
print("B is waiting at:", review.get_state(b).next)

print("\napprove A, reject B")
print("A:", review.invoke(Command(resume="approve"), a)["result"])
print("B:", review.invoke(Command(resume="reject"), b)["result"])

print("\nthe saved state of A is still there:", review.get_state(a).values)

A is waiting at: ('gate',)
B is waiting at: ('gate',)

approve A, reject B
A: applied: refund 20 dollars
B: nothing was done

the saved state of A is still there: {'proposal': 'refund 20 dollars', 'decision': 'approve', 'result': 'applied: refund 20 dollars'}


## Where to look next in the project

| Concept | File |
|---|---|
| State, reducer, nodes, edges, parallel, conditional edges | `backend/src/triage/graph.py` |
| Interrupt (the approval gate) | the `gate` node in `backend/src/triage/agents.py` |
| Resume and thread id | `RunManager.decide` and `RunManager.start` in `backend/src/triage/runs.py` |